# 문서별 `-었-` 결합률·로그오즈비 생성

`word_ver8.7_sen.csv`를 바탕으로 문서별 결합 통계를 만든다.

- 분석 대상: `speaker == 'body'`, `sent_has_E == True`, `last_EN_in_quote == False`
- 단순과거 결합: `았|었`을 포함하되 중첩 과거 `았었|었었`은 제외
- 오즈비: 해당 문서 대 나머지 전체 문서의 one-vs-rest 2×2 오즈비
- 2×2 표에 0이 있으면 네 칸 모두에 0.5를 더해 보정
- 분석 대상 문장이 없는 문서는 오즈비와 로그오즈비를 0으로 처리


In [1]:
from pathlib import Path
from datetime import datetime
import math

import numpy as np
import pandas as pd

SEN_CSV = Path(r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.7_sen.csv")

# 기존 세종문어_document_정보2.csv가 파일 단위 자료로 덮어써진 상태이므로
# 우선 별도 파일로 저장한다. 확인 후 파일명을 바꿔도 된다.
OUTPUT_CSV = Path(r"..\csv\original_csv\세종문어_document_었결합률.csv")


In [2]:
required_cols = {
    "docu_id", "sen_id", "speaker", "sent_has_E",
    "last_EN_in_quote", "sentence_f_EP_form",
}

df_sen = pd.read_csv(SEN_CSV, low_memory=False)
missing_cols = sorted(required_cols - set(df_sen.columns))
if missing_cols:
    raise ValueError(f"필수 컬럼이 없습니다: {missing_cols}")

df_sen = df_sen.dropna(subset=["docu_id"]).copy()
df_sen["docu_id"] = df_sen["docu_id"].astype("string").str.strip()

print(f"문장 파일: {SEN_CSV}")
print(f"문장 수: {len(df_sen):,}")
print(f"문서 수: {df_sen['docu_id'].nunique():,}")


문장 파일: ..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.7_sen.csv
문장 수: 1,048,126
문서 수: 33,155


In [3]:
def as_bool(series: pd.Series) -> pd.Series:
    """bool·숫자·문자열로 저장된 논릿값을 안전하게 bool로 변환한다."""
    if pd.api.types.is_bool_dtype(series.dtype):
        return series.fillna(False).astype(bool)

    normalized = series.astype("string").str.strip().str.lower()
    return normalized.isin(["true", "1", "t", "yes", "y"])


sent_has_e = as_bool(df_sen["sent_has_E"])
last_en_in_quote = as_bool(df_sen["last_EN_in_quote"])
speaker = df_sen["speaker"].astype("string").str.strip().str.lower()
ep_form = df_sen["sentence_f_EP_form"].astype("string").str.replace(" + ", "", regex=False)

is_head = speaker.eq("head").fillna(False)
is_body = speaker.eq("body").fillna(False)
body_has_e = is_body & sent_has_e
body_not_quote = body_has_e & ~last_en_in_quote

# 단순과거: 았/었은 포함하되 았었/었었은 별도 중첩 과거이므로 제외한다.
is_double_past = ep_form.str.contains(r"(?:았었|었었)", regex=True, na=False)
is_simple_past = (
    ~is_double_past
    & ep_form.str.contains(r"(?:았|었)", regex=True, na=False)
)
body_not_quote_and_past = body_not_quote & is_simple_past

df_sen["_is_head"] = is_head.astype("int8")
df_sen["_is_body"] = is_body.astype("int8")
df_sen["_body_has_E"] = body_has_e.astype("int8")
df_sen["_body_not_quote"] = body_not_quote.astype("int8")
df_sen["_body_not_quote_and_었"] = body_not_quote_and_past.astype("int8")


In [4]:
df_docu_ = (
    df_sen.groupby("docu_id", sort=False, observed=True)
    .agg(
        sent_count=("docu_id", "size"),
        head_count=("_is_head", "sum"),
        body_count=("_is_body", "sum"),
        body_has_E_count=("_body_has_E", "sum"),
        body_not_quote_count=("_body_not_quote", "sum"),
        body_not_quote_and_었_count=("_body_not_quote_and_었", "sum"),
    )
    .reset_index()
)

count_cols = [
    "sent_count", "head_count", "body_count", "body_has_E_count",
    "body_not_quote_count", "body_not_quote_and_었_count",
]
df_docu_[count_cols] = df_docu_[count_cols].astype("int64")

denominator = df_docu_["body_not_quote_count"]
numerator = df_docu_["body_not_quote_and_었_count"]
df_docu_["었_결합률"] = np.divide(
    numerator,
    denominator,
    out=np.zeros(len(df_docu_), dtype="float64"),
    where=denominator.ne(0),
)

df_docu_.head()


,docu_id,sent_count,head_count,body_count,body_has_E_count,body_not_quote_count,body_not_quote_and_었_count,었_결합률
0,AA0001.000,2,0,2,0,0,0,0.000000
1,AA0001.001,10,1,9,8,7,3,0.428571
2,AA0001.002,18,2,16,13,13,0,0.000000
3,AA0001.003,17,1,16,12,12,2,0.166667
4,AA0001.004,15,2,13,12,12,1,0.083333


In [5]:
def one_vs_rest_odds(
    group_success: int,
    group_total: int,
    global_success: int,
    global_total: int,
    alpha: float = 0.5,
) -> tuple[float, float]:
    """해당 문서 대 나머지 문서의 오즈비와 자연로그 오즈비를 반환한다."""
    if group_total == 0:
        return 0.0, 0.0

    a = float(group_success)
    b = float(group_total - group_success)
    c = float(global_success - group_success)
    d = float((global_total - global_success) - b)

    if any(cell == 0 for cell in (a, b, c, d)):
        a, b, c, d = (cell + alpha for cell in (a, b, c, d))

    odds_ratio = (a * d) / (b * c)
    return odds_ratio, math.log(odds_ratio)


global_total = int(df_docu_["body_not_quote_count"].sum())
global_success = int(df_docu_["body_not_quote_and_었_count"].sum())

odds_result = [
    one_vs_rest_odds(
        group_success=int(success),
        group_total=int(total),
        global_success=global_success,
        global_total=global_total,
    )
    for success, total in zip(
        df_docu_["body_not_quote_and_었_count"],
        df_docu_["body_not_quote_count"],
    )
]

df_docu_["었_결합_오즈비"] = [value[0] for value in odds_result]
df_docu_["었_결합_로그오즈비"] = [value[1] for value in odds_result]

final_cols = [
    "docu_id", "sent_count", "head_count", "body_count",
    "body_has_E_count", "body_not_quote_count",
    "body_not_quote_and_었_count", "었_결합률",
    "었_결합_오즈비", "었_결합_로그오즈비",
]
df_docu_ = df_docu_[final_cols]

print(f"전체 분석 대상 문장: {global_total:,}")
print(f"전체 단순과거 결합 문장: {global_success:,}")
print(f"전체 단순과거 결합률: {global_success / global_total:.6%}")
df_docu_.head()


전체 분석 대상 문장: 843,110
전체 단순과거 결합 문장: 353,458
전체 단순과거 결합률: 41.923118%


,docu_id,sent_count,head_count,body_count,body_has_E_count,body_not_quote_count,body_not_quote_and_었_count,었_결합률,었_결합_오즈비,었_결합_로그오즈비
0,AA0001.000,2,0,2,0,0,0,0.000000,0.000000,0.000000
1,AA0001.001,10,1,9,8,7,3,0.428571,1.038989,0.038249
2,AA0001.002,18,2,16,13,13,0,0.000000,0.051307,-2.969934
3,AA0001.003,17,1,16,12,12,2,0.166667,0.277060,-1.283522
4,AA0001.004,15,2,13,12,12,1,0.083333,0.125936,-2.071985


In [6]:
assert df_docu_["docu_id"].is_unique
assert (df_docu_["body_not_quote_and_었_count"] <= df_docu_["body_not_quote_count"]).all()
assert df_docu_["었_결합률"].between(0, 1).all()

eligible = df_docu_["body_not_quote_count"].gt(0)
assert df_docu_.loc[eligible, "었_결합_오즈비"].gt(0).all()
assert np.allclose(
    np.log(df_docu_.loc[eligible, "었_결합_오즈비"]),
    df_docu_.loc[eligible, "었_결합_로그오즈비"],
)

print(f"검증 완료: {len(df_docu_):,}개 문서")
df_docu_.describe(include="all")


검증 완료: 33,155개 문서


,docu_id,sent_count,head_count,body_count,body_has_E_count,body_not_quote_count,body_not_quote_and_었_count,었_결합률,었_결합_오즈비,었_결합_로그오즈비
count,33155,33155.000000,33155.000000,33155.000000,33155.000000,33155.000000,33155.000000,33155.000000,33155.000000,33155.000000
unique,33155,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,AA0001.003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,31.612909,1.238697,30.374122,29.134158,25.429347,10.660775,0.383223,2.548399,-0.373441
std,NaN,121.369717,0.540314,121.402467,117.991827,104.773241,55.949400,0.344624,4.879260,1.718237
min,NaN,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-5.579806
25%,NaN,7.000000,1.000000,5.000000,5.000000,4.000000,1.000000,0.062500,0.173163,-1.666536
50%,NaN,14.000000,1.000000,13.000000,12.000000,11.000000,3.000000,0.300000,0.593701,-0.485007
75%,NaN,25.000000,1.000000,23.000000,23.000000,21.000000,6.000000,0.666667,2.770648,1.019081


In [7]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_docu_.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"저장 완료: {OUTPUT_CSV}")
print(f"완료 시각: {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")


저장 완료: ..\csv\original_csv\세종문어_document_었결합률.csv
완료 시각: 2026.07.01_22:12:59
